# Herding simulations: Jadhav, Di Lorenzo and the hybrids

Every simulation has a **time slider**. **Play** runs the whole simulation in 15 s, so its end goal (reaching the target, or settling at the target density) comes up within 15 s. Drag the slider to any moment, or use the **Jump to** buttons for the key moments.

The sliders are saved in this notebook and play without running anything. Code cells are collapsed: click the bar to the left of a cell to open one.

| § | Simulation | Sheep | Herders | Space | Status |
|---|---|---|---|---|---|
| 1 | Jadhav et al. 2024 | 14, published heading rule | 1 Strömbom dog | open plane | reference (authors' code values: 4 attracting neighbours, noise 0.3, dog notice radius 12 m), continued past the published 370 s |
| 2.1 | Di Lorenzo, published scale | 720, first-order | 280, designed | periodic 2π box | reference (push kernel with 1 periodic image) |
| 2.2 | Di Lorenzo in metres | 36, first-order | 14, designed | 24 m periodic box | adaptation (rescaled; 36 + 14 agents) |
| 2.2 | Follower swap | 36, published Jadhav sheep | the same 14 | the same box | hybrid |
| 2.3 | Jadhav on Di Lorenzo's torus | 14, published heading rule | 1 Strömbom dog | periodic box, size adjustable (24 m is Di Lorenzo's) | adaptation (space and starts changed) |
| 3 | Hybrid A | 60, heading rule, always active, Brownian D = 0.5 m²/s | 16, designed for a weak push, capped at 1.5 m/s | 80 m periodic box | hybrid |
| 4 | Hybrid B | 50, first-order, Morse cohesion | 1 Strömbom dog | open plane | hybrid |

Settings differ between sections, so results compare only within a section. Section 5 checks this notebook's numbers against the slides.

In [1]:
import json
import sys
from dataclasses import replace
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.sparse.csgraph import connected_components

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "shepherd").is_dir())
sys.path[:0] = [str(ROOT), str(ROOT / "talk"), str(ROOT / "experiments")]

from shepherd import bridge as B
from shepherd import metrics as M
from shepherd.model import COLLECT, DRIVE, SLOW, Params, simulate
from shepherd.scorecard import _episodes, score
from slider_player import GOAL, HERDER, SHEEP, choice_player, panel, player, strip, variant
import closed_loop_convergence as CL

PLAY_SECONDS = 15  # default 15: Play runs the whole simulation in this many seconds
SEED = 0           # default 0: the run shown on the sliders after section 1 (section 1 has its own Seed slider)
NATIVE, ORANGE = "#1565C0", "#E65100"
pd.set_option("display.max_colwidth", None)
CHECKS = []        # one row per number compared in section 5


def stored(name):
    return json.loads((ROOT / "results" / f"{name}.json").read_text())


def check(simulation, quantity, value, reference, tolerance, slide):
    CHECKS.append(dict(simulation=simulation, quantity=quantity, notebook=round(float(value), 3), stored=round(float(reference), 3),
                       slide=slide, match="yes" if abs(value - reference) <= tolerance else "NO"))


def run(cfg, seed=SEED, store_every=1, **over):
    F, H, T, D, kw = cfg
    return B.simulate_bridge(F, H, T, D, seed=seed, store_every=store_every, **{**kw, **over})


def settled_time(t, y, window=10, band=0.1):
    # Settling time, a navigation marker: first time the running mean of y has covered 1 - band of the way from
    # its starting value to its last-quarter mean and stays there.
    y = np.asarray(y, float)
    mean = np.convolve(y, np.ones(window) / window, "valid")
    tail = y[-len(y) // 4:].mean()
    outside = np.flatnonzero(np.abs(mean - tail) > band * abs(y[0] - tail))
    k = 0 if len(outside) == 0 else outside[-1] + 1
    return None if k >= len(mean) else float(t[k + window - 1])


def groups(x, D, link=5.0, min_size=3):
    # Number of groups of at least min_size sheep (linked within `link` m), and the largest group's share.
    d = np.linalg.norm(D.wrap(x[:, None] - x[None]), axis=-1)
    sizes = np.bincount(connected_components(d < link, directed=False)[1])
    return int((sizes >= min_size).sum()), sizes.max() / len(x)


def target_heat(T, D, half=None):
    # The target density the sheep should reach, drawn as the grey background: darkest at the goal, fading outward.
    half = D.size / 2 if half is None else half
    return dict(grid=B.Grid(2 * half, 64 if D.periodic else 128).target(T.resolve(D), D)[0].T, extent=(-half, half, -half, half))


def bridge_panel(r, title, half=None, **kw):
    D, T = r.domain, r.target.resolve(r.domain)
    half = D.size / 2 if half is None else half
    heat = target_heat(T, D, half)
    caption = [f"inside goal {100 * o:.0f}%     density error {e:.2f}" for o, e in zip(r.metrics["occupancy"], r.metrics["density_error"])]
    return panel(title, D.wrap(r.pos_s), D.wrap(r.pos_d), (-half, half), (-half, half), heat=heat, caption=caption,
                 goal=dict(x=T.centre[0], y=T.centre[1], r=T.r_goal), **kw)


def occupancy_strip(runs):
    return strip("Share of sheep inside the goal circle", [(lab, col, r.metrics["occupancy"]) for lab, col, r in runs], 0, 1, hlines=[(0.8, "success 0.8")])


def error_strip(runs):
    top = 1.05 * max(np.nanmax(r.metrics["density_error"]) for _, _, r in runs)
    return strip("Density error (0 = target)", [(lab, col, r.metrics["density_error"]) for lab, col, r in runs], 0, top, hlines=[(runs[0][2].floor, "floor")])

## 1. Jadhav's published model

`shepherd/model.py` at the published parameters; the port matches the authors' MATLAB trajectories to 10⁻⁸ over 30 steps. One dog drives 14 sheep from 250 m away.

**Length:** the published run stops at 370 s. This one continues to 500 s under the same rules, to show what the dog does after arrival. The first 370 s are unchanged.

**Seed:** the upper slider changes the random seed. The default, **seed 11**, is the run in `talk/demos/jadhav_replication.mp4`. As for that video, each seed simulates 40 drives and shows the one arriving closest to t = 300 s, so the arrival time looks alike across seeds by construction. To try other seeds, edit `JADHAV_SEEDS` in this section's code cell and run it.

**End goal:** the flock centre comes within 8 m of the target.

**Watch for:** scattered sheep form one flock, the dog settles behind it, and the flock moves straight to the target. Then it passes the target and the dog keeps driving, because its rule has no stop condition: use the third panel, which stays fixed on the target. In the middle panel the dashed circle is 12 m around the dog and the ring marks the farthest sheep.

In [2]:
JADHAV_SEED = 11          # default 11: the run in talk/demos/jadhav_replication.mp4
JADHAV_SEEDS = range(12)  # default 0 to 11: the seeds on the Seed slider, one run each
JADHAV_SECONDS = 500      # default 500: the published run stops at 370 s; the first 370 s do not change
PUBLISHED_SECONDS = 370
MODES = {DRIVE: "drive", COLLECT: "collect", SLOW: "slow (close to a sheep)"}


def jadhav_variant(seed):
    p = Params(n_steps=JADHAV_SECONDS)
    N = p.n_sheep
    runs = simulate(p, n_runs=40, seed=seed)
    to_target = np.linalg.norm(runs.pos_s.mean(axis=1), axis=1)
    arrive = np.array([np.argmax(d < 8) if (d < 8).any() else 10 ** 6 for d in to_target])
    pick = int(np.argmin(np.abs(arrive - 300)))  # as in talk/make_media.py: the run arriving closest to t = 300 s
    reached = int(arrive[pick])
    pos, vel = runs.run(pick)
    mode = runs.mode[pick]
    b, C, P, E = M.barycenter(pos, N), M.cohesion(pos, N), M.polarization_from_vel(vel, N), M.frame_observables(pos, vel, N)[0]
    spread = np.linalg.norm(pos[:N] - b[None], axis=1)
    R, farthest, dist = spread.max(axis=0), spread.argmax(axis=0), np.linalg.norm(b, axis=0)
    sheep, dog, headings, t = pos[:N].transpose(2, 0, 1), pos[N].T[:, None], vel[:N].transpose(2, 0, 1), np.arange(pos.shape[2])
    dog_dist = np.linalg.norm(pos[N], axis=0)
    walked = np.r_[0.0, np.cumsum(np.linalg.norm(np.diff(pos[N], axis=1), axis=0))]
    since = walked - walked[reached]
    f_N = p.r_rep * N ** (2 / 3)

    lo, hi = np.minimum(b.min(axis=1), 0), np.maximum(b.max(axis=1), 0)
    mid, half = (lo + hi) / 2, (hi - lo).max() / 2 + 30
    trails = [(b.T, SHEEP, 2), (pos[N].T, HERDER, 0.8)]
    whole = panel("Whole drive", sheep, dog, (mid[0] - half, mid[0] + half), (mid[1] - half, mid[1] + half), decimals=1,
                  goal=dict(x=0, y=0, r=8, fill=True, label="target"), trails=trails,
                  caption=[f"dog: {MODES.get(int(m), 'starting')}" for m in mode], scale=50, scale_label="50 m", sheep_px=1.8, dog_px=6)
    follow = panel("Following the flock", None, None, (-22, 22), (-22, 22), view=dict(cx=b[0], cy=b[1], half=22), goal=dict(x=0, y=0, r=8, fill=True),
                   headings=headings, heading_len=1.4, dog_ring=p.r_dog, highlight=farthest, scale=10, scale_label="10 m", sheep_px=4.5,
                   caption=[f"P = {P[i]:.2f}   C = {C[i]:.1f} m   E = {E[i]:.2f}   R = {R[i]:.1f} m" for i in t])
    near = float(max(40.0, np.ceil(max(dist[reached:].max(), dog_dist[reached:].max()) + 10)))
    around = panel("Around the target (fixed view)", None, None, (-near, near), (-near, near), goal=dict(x=0, y=0, r=8, fill=True, label="target"), trails=trails,
                   scale=20, scale_label="20 m", sheep_px=2.8, dog_px=8,
                   caption=["flock has not reached the target yet" if i < reached else f"dog has moved {since[i]:.0f} m since the flock reached it" for i in t])
    strips = [strip("Distance to target (m)", [("flock centre", SHEEP, dist), ("dog", HERDER, dog_dist)], 0, 1.02 * max(dist.max(), dog_dist.max()), hlines=[(8, "8 m")]),
              strip("Polarization P", [("P", SHEEP, P)], 0, 1),
              strip("Cohesion C and farthest sheep R (m)", [("C", SHEEP, C), ("R", HERDER, R)], 0, max(R.max(), f_N) * 1.05, hlines=[(f_N, "dog collects above")])]
    after = dist[reached:]
    events = sorted([(reached, "Reaches the target"), (reached + int(after[:60].argmin()), "Closest to the target"), (PUBLISHED_SECONDS - 1, "Published run ends"),
                     (reached + int(after.argmax()), f"Farthest past it ({after.max():.0f} m)")])
    note = (f"Seed {seed}, run {pick} of 40. The dog moves {since[-1]:.0f} m between the flock reaching the target and the end of the run." + (" Default seed." if seed == JADHAV_SEED else ""))
    return variant(t, [whole, follow, around], strips, events, note=note, play_seconds=PLAY_SECONDS, pace=[(reached, 8)])


seeds = list(JADHAV_SEEDS)
choice_player("Jadhav's published model: 14 sheep, 1 dog", "Seed", [(f"{s} (default)" if s == JADHAV_SEED else str(s), jadhav_variant(s)) for s in seeds],
              start=seeds.index(JADHAV_SEED) if JADHAV_SEED in seeds else 0)

**After arrival.** The dog's rule has no stop condition, so reaching the target does not end the drive. The ensemble below is the one behind the slide: 200 runs started 100 m out, 300 steps, "reached" meaning within 5 m (`experiments/closed_loop_convergence.py`).

In [3]:
res = simulate(CL.base(), n_runs=CL.R, seed=CL.SEED)
g = CL.geometry(res)
post = CL.post_reach(g, res.mode, CL.reach_times(g))
check("Jadhav", "mean distance from target after reaching (m)", post["rho_mean"], stored("closed_loop")["post_reach"]["rho_mean"], 0.1, "about 20 m")
print(f"After reaching, the flock centre averages {post['rho_mean']:.1f} m from the target ({post['n_runs']} runs).")

After reaching, the flock centre averages 19.6 m from the target (200 runs).


## 2. Di Lorenzo comparison

### 2.1 Their published setting

Di Lorenzo et al. (arXiv:2411.04791, Sec. IV-A), in their units: 720 first-order followers and 280 herders on a periodic [−π, π]² box, D = 0.01, push range L = π, K_H = 10, von Mises goal k = 6/π with r* = π/2, dt = 0.01, T = 200. The herder density is designed once, from the follower model, by deconvolution; the herders then track it with a density flow.

The grey background is the target density for the followers, darkest at the goal. This run takes about 13 minutes, so it loads from `talk/results/dilorenzo_reference.npz`. Set `RERUN = True` to recompute it.

**End goal:** convergence to the target density, marked where the density error is 90% of the way from its start to its final level. Occupancy is not the test here: the target itself keeps only about 76% of its mass inside r*.

In [4]:
RERUN = False  # default False: load the cached run; True recomputes it (about 13 minutes)
CACHE = ROOT / "talk" / "results" / "dilorenzo_reference.npz"
F, H, T, D, kw = B.preset("dilorenzo_reference")
if RERUN or not CACHE.exists():
    r = run((F, H, T, D, kw), store_every=200)
    s = score(r)
    np.savez_compressed(CACHE, t=r.t, pos_s=D.wrap(r.pos_s).astype("f4"), pos_d=D.wrap(r.pos_d).astype("f4"), rho_H=r.design.rho_H.astype("f4"),
                        mass_needed=r.design.mass_needed, mass_available=r.design.mass_available, floor=r.floor, r_goal=T.resolve(D).r_goal,
                        seed=SEED, images=D.images, time_to_success=np.nan if s["time_to_success"] is None else s["time_to_success"],
                        final_occupancy=s["final_occupancy"], **{f"m_{k}": v for k, v in r.metrics.items()})
z = np.load(CACHE)
t_ref, occ, err = z["t"], z["m_occupancy"], z["m_density_error"]
L = np.pi
ref_panel = panel("720 followers (dots), 280 herders (triangles)", z["pos_s"], z["pos_d"], (-L, L), (-L, L), goal=dict(x=0, y=0, r=float(z["r_goal"])),
                  heat=target_heat(T, D), scale=1, scale_label="1 length unit", sheep_px=1.5, dog_px=3.6,
                  caption=[f"inside goal {100 * o:.0f}%     density error {e:.2f}" for o, e in zip(occ, err)])
strips = [strip("Share of followers inside the goal circle", [("occupancy", SHEEP, occ)], 0, 1),
          strip("Density error (0 = target)", [("error", SHEEP, err)], 0, 1.05 * err.max(), hlines=[(float(z["floor"]), "floor")])]
settled = settled_time(t_ref, err, window=5)
events = [] if settled is None else [(settled, "Density error 90% settled")]
print(f"Herder mass available {float(z['mass_available']):.3f}, needed M̂ = {float(z['mass_needed']):.3f} (push kernel with {int(z['images'])} periodic image; "
      f"M̂ grows with the image count, docs/bridge_simulation.md). Last-quarter density error {err[-len(err) // 4:].mean():.3f}, floor {float(z['floor']):.3f}; "
      f"share inside r* {occ[-len(occ) // 4:].mean():.2f}.")
player("Di Lorenzo's published setting", t_ref, [ref_panel], strips, events, unit="time units", play_seconds=PLAY_SECONDS)

Herder mass available 0.280, needed M̂ = 0.241 (push kernel with 1 periodic image; M̂ grows with the image count, docs/bridge_simulation.md). Last-quarter density error 0.011, floor 0.011; share inside r* 0.78.


### 2.2 The same herders with two follower models (the follower swap)

Same seed and same starts. **Fixed:** von Mises target, goal circle r* = 6 m, 24 m periodic box, push range L = 12 m, 14 herders and their flow (K_H = 10, no speed cap, 100 sub-steps per second), and the herder density, computed once from **Di Lorenzo's** follower model. **Changed:** only the follower update. Left: their first-order drift plus Brownian noise (36 followers). Right: 36 published Jadhav sheep.

**End goal:** at least 80% of the sheep inside the goal circle for 30 s (the scorecard's definition), and the density error against its floor.

**Occupancy** measures confinement and **density error** measures density matching. Passing one does not imply the other: the target itself has only 76% of its mass inside the goal circle. Grey is the target density.

In [5]:
native = run(B.preset("dilorenzo_metres"), store_every=10)  # dt = 0.1 s: one frame per second
swap = run(B.preset("dilorenzo_jadhav_sheep"))
both = [("native followers", NATIVE, native), ("Jadhav sheep", SHEEP, swap)]
panels = [bridge_panel(r, title, scale=5, scale_label="5 m", sheep_px=4.5, dog_px=8) for title, r in (("Native followers (Di Lorenzo)", native), ("Published Jadhav sheep", swap))]
cohesion = strip("Cohesion (m)", [(lab, col, r.metrics["cohesion"]) for lab, col, r in both], 0, 1.05 * max(np.nanmax(r.metrics["cohesion"]) for *_, r in both))
events = [(score(r)["time_to_success"], f"{lab}: 80% inside for 30 s") for lab, _, r in both if score(r)["time_to_success"] is not None]
player("Same herders, same target, same starts", swap.t, panels, [occupancy_strip(both), error_strip(both), cohesion], events, play_seconds=PLAY_SECONDS)

**Matched comparison** (5 seeds, 200 s, last quarter), with the same herders under a flat design as a control. This cell takes about a minute and is only needed to re-derive the numbers.

In [6]:
F, H, T, D, kw = cfg_swap = B.preset("dilorenzo_jadhav_sheep")
cfgs = {"Native followers": B.preset("dilorenzo_metres"), "Jadhav sheep": cfg_swap,
        "Jadhav sheep, flat design": (F, replace(H, design_model=replace(H.design_model, D=0.0)), T, D, kw)}
matched = {}
for name, cfg in cfgs.items():
    tails = [run(cfg, seed=s, store_every=5).tail_mean(0.25) for s in range(5)]
    matched[name] = {k: float(np.mean([m[k] for m in tails])) for k in ("centroid_error", "occupancy", "density_error", "cohesion")}
table = pd.DataFrame(matched).T.rename(columns={"centroid_error": "centroid error (m)", "occupancy": "occupancy", "density_error": "density error", "cohesion": "cohesion (m)"})
display(table.round(2))
saved = stored("dilorenzo_jadhav_sheep")["rows"]
for name, key, slide_vals in (("Native followers", "reference", ("86%", "0.24", "0.7 m")), ("Jadhav sheep", "jadhav", ("100%", "1.29", "0.4 m"))):
    check(f"Follower swap, {name}", "occupancy", matched[name]["occupancy"], saved[key]["occupancy"], 0.01, slide_vals[0])
    check(f"Follower swap, {name}", "density error", matched[name]["density_error"], saved[key]["density_error"], 0.02, slide_vals[1])
    check(f"Follower swap, {name}", "centroid error (m)", matched[name]["centroid_error"], saved[key]["centroid_error"], 0.05, slide_vals[2])
check("Follower swap, flat design", "occupancy", matched["Jadhav sheep, flat design"]["occupancy"], saved["undesigned"]["occupancy"], 0.01, "17%")
check("Follower swap, flat design", "centroid error (m)", matched["Jadhav sheep, flat design"]["centroid_error"], saved["undesigned"]["centroid_error"], 0.1, "10 m")

,centroid error (m),occupancy,density error,cohesion (m)
Native followers,0.68,0.86,0.24,3.62
Jadhav sheep,0.44,1.00,1.29,1.88
"Jadhav sheep, flat design",10.03,0.17,1.75,5.30


**Expanded validation, Jadhav sheep only.** A separate experiment, read from `results/follower_swap_validation.json` and not re-run here: 50 fresh seeds for 600 s (late window 450–600 s) and six runs to 6000 s.

In [7]:
v = stored("follower_swap_validation")
within3 = np.mean([s["late"]["occ_by_radius"]["3.0"] for s in v["short"]])
blocks = np.array([[b["timeavg_err"] for b in r["blocks"]] for r in v["long"]])
print(f"{v['summary']['full_late']}/{len(v['short'])} runs kept full occupancy throughout the late window.")
print(f"{100 * within3:.0f}% of the sheep lie within 3 m of the goal centre; the target requests {100 * v['target']['mass_by_radius']['3.0']:.0f}%.")
means = blocks.mean(axis=0)
print(f"Six long runs: time-averaged density error per 600 s block, mean over runs: {means[0]:.2f} in the first block (includes the start), "
      f"{means[1:].min():.2f} to {means[1:].max():.2f} after it (floor {stored('dilorenzo_jadhav_sheep')['rows']['jadhav']['floor']:.2f}).")

47/50 runs kept full occupancy throughout the late window.
86% of the sheep lie within 3 m of the goal centre; the target requests 33%.
Six long runs: time-averaged density error per 600 s block, mean over runs: 0.94 in the first block (includes the start), 1.03 to 1.06 after it (floor 0.22).


**What the follower swap supports.** The designed herders reliably confined Jadhav sheep in the tested setting, but did not reproduce the target density. The validation used one box size, one herder count and one herder design, and why the density differs from the target has not been established.

### 2.3 Jadhav's model on Di Lorenzo's torus

Jadhav's published sheep and dog, unchanged (14 sheep, one Strömbom dog, the authors' code values), moved into the box of section 2.2: 24 m periodic box, goal circle r* = 6 m, uniform starts, 900 s. Only the space and the starts change. Opposite edges are joined, so a flock leaving on the right comes back on the left. The run uses `shepherd/bridge.py`, which has the periodic domain and reproduces the published model on the plane (`tests/test_bridge.py`).

**End goal:** at least 80% of the sheep inside the goal circle for 30 s, as in section 2.2.

**Box side:** the upper slider changes the size of the periodic box, and the time slider keeps its place when you move it. It starts at 24 m, Di Lorenzo's box. The goal circle stays at 6 m and the target density stays the same in metres, so only the space changes. To try other sizes, edit `BOX_SIZES` in this section's code cell and run it (a few seconds).

**Watch for:** the dog's rule still has no stop condition, so the flock passes through the goal. On the torus it comes round again instead of walking away. Play slows down for the flock forming and for its first arrival. Grey is the same target density as section 2.2; it is only used for scoring, since this dog steers toward the goal's centre and does not read the density.

In [8]:
JADHAV_TORUS_SECONDS = 900  # default 900: long enough for every seed's first arrival in the 24 m box
BOX_SIZES = [16, 20, 24, 26, 28, 30, 36, 48, 60, 80]  # default list, box side in metres, one run each; the slider starts at 24 m
_, _, T_box, D_box, _ = B.preset("dilorenzo_jadhav_sheep")  # section 2.2's box and goal


def box(side):
    # A periodic box of this side, with section 2.2's goal circle and the same target density in metres.
    return B.Domain(True, float(side), images=D_box.images), replace(T_box, k=T_box.k * (side / D_box.size) ** 2)


def jadhav_in(domain=D_box, target=T_box, n_sheep=14):
    return B.Followers(n=n_sheep), B.Herders("strombom", n=1), target, domain, dict(init="uniform", n_steps=JADHAV_TORUS_SECONDS)


def goal_share(side):
    return np.pi * T_box.resolve(D_box).r_goal ** 2 / side ** 2


def torus_variant(side):
    r = run(jadhav_in(*box(side)))
    one = [("occupancy", SHEEP, r)]
    visits = _episodes(r.metrics["occupancy"] >= 0.8, r.t, 30.0)
    pt = bridge_panel(r, f"14 sheep, 1 dog, {side} m periodic box (opposite edges are joined)", headings=r.hd_s, heading_len=1.2, scale=5, scale_label="5 m",
                      sheep_px=4.5 if side <= 30 else 3.2, dog_px=9 if side <= 30 else 7)
    strips = [occupancy_strip(one), error_strip(one), strip("Flock centre to goal (m)", [("distance", SHEEP, r.metrics["centroid_error"])], 0, 1.05 * r.metrics["centroid_error"].max())]
    first = visits[0][0] if visits else None
    if first is None:
        pace = [(120, 8)]
    elif first > 80:
        pace = [(60, 3), (first - 17, 5), (min(first + 53, r.t[-1] - 1), 11)]
    else:
        pace = [(first + 53, 8)]
    note = f"Seed {SEED}. The goal circle covers {goal_share(side):.0%} of this box." + ("" if visits else " The end goal is not reached in this run.")
    return variant(r.t, [pt], strips, [(v[0], f"Visit {i + 1}: 80% inside") for i, v in enumerate(visits[:4])], note=note, play_seconds=PLAY_SECONDS, pace=pace)


choice_player("Jadhav's model on Di Lorenzo's torus", "Box side", [(f"{s} m" + (" (default, Di Lorenzo)" if s == D_box.size else ""), torus_variant(s)) for s in BOX_SIZES],
              start=BOX_SIZES.index(24) if 24 in BOX_SIZES else 0)

**Five seeds, 900 s**, averaged after the first 100 s. A pass is any stretch with at least 80% of the sheep inside the goal circle; the end goal needs one lasting 30 s. The 36-sheep row uses section 2.2's sheep count. The plane row keeps the same starts and goal circle and removes the wrap-around; its density error is left out because that target is not defined on the plane.

In [9]:
def jadhav_row(cfg, density=True):
    rows = []
    for s in range(5):
        r = run(cfg, seed=s)
        o, w = r.metrics["occupancy"], r.t >= 100
        passes = [d for _, d, _ in _episodes(o >= 0.8, r.t, 1.0)]
        rows.append(dict(reached=score(r)["time_to_success"], visits=len(_episodes(o >= 0.8, r.t, 30.0)), passes=len(passes), longest=max(passes, default=0.0), occupancy=o[w].mean(),
                         error=r.metrics["density_error"][w].mean(), floor=r.floor, cohesion=r.metrics["cohesion"][w].mean(), centre=r.metrics["centroid_error"][w].mean()))
    reached = [x["reached"] for x in rows if x["reached"] is not None]
    mean = lambda k: float(np.mean([x[k] for x in rows]))
    return {"seeds meeting the end goal": f"{len(reached)} of {len(rows)}", "first met (s)": f"{min(reached):.0f} to {max(reached):.0f}" if reached else "never",
            "passes per run": round(mean("passes"), 1), "longest pass (s)": round(max(x["longest"] for x in rows)),
            "visits of 30 s or more": f"{min(x['visits'] for x in rows)} to {max(x['visits'] for x in rows)}", "occupancy after 100 s": round(mean("occupancy"), 2),
            "density error": round(mean("error"), 2) if density else np.nan, "floor": round(mean("floor"), 2) if density else np.nan,
            "cohesion (m)": round(mean("cohesion"), 1), "centre to goal (m)": round(mean("centre"), 1)}


plane = B.Domain(False, D_box.size)  # the size only sets the uniform starts
plane_target = B.Target("gaussian", sigma=2.5, r_goal=T_box.resolve(D_box).r_goal)
table = pd.DataFrame({"14 sheep, torus": jadhav_row(jadhav_in()), "36 sheep, torus": jadhav_row(jadhav_in(n_sheep=36)),
                      "14 sheep, open plane": jadhav_row(jadhav_in(plane, plane_target), density=False)}).T
display(table)
f = B.Followers(n=36)
print(f"The goal circle covers {goal_share(D_box.size):.0%} of the 24 m box. The farthest any point can be from the flock centre is {D_box.size / np.sqrt(2):.1f} m; "
      f"with 36 sheep the dog collects only beyond {f.r_rep * 36 ** (2 / 3):.1f} m, so it never collects.")

,seeds meeting the end goal,first met (s),passes per run,longest pass (s),visits of 30 s or more,occupancy after 100 s,density error,floor,cohesion (m),centre to goal (m)
"14 sheep, torus",5 of 5,23 to 620,38.2,58,2 to 7,0.66,1.55,0.57,2.8,4.4
"36 sheep, torus",0 of 5,never,5.2,8,0 to 0,0.39,1.04,0.22,5.2,5.6
"14 sheep, open plane",0 of 5,never,9.4,12,0 to 0,0.14,NaN,NaN,2.9,18.6


The goal circle covers 20% of the 24 m box. The farthest any point can be from the flock centre is 17.0 m; with 36 sheep the dog collects only beyond 21.8 m, so it never collects.


**Box size, 14 sheep** (5 seeds, 900 s, averaged after the first 100 s). The goal circle takes up less of the box as the box grows, so compare occupancy with its share of the box. Density error is left out of this table: the slider shows it, computed on a 64-point grid at every size, which stays within 7% of a 384-point grid up to 80 m.

In [10]:
sizes = pd.DataFrame({f"{s} m": {"goal circle's share of the box": f"{goal_share(s):.0%}", **jadhav_row(jadhav_in(*box(s)))} for s in BOX_SIZES}).T.drop(columns=["density error", "floor"])
display(sizes)
meeting = [s for s in BOX_SIZES if sizes.loc[f"{s} m", "seeds meeting the end goal"] == "5 of 5"]
none = [s for s in BOX_SIZES if sizes.loc[f"{s} m", "seeds meeting the end goal"] == "0 of 5"]
print(f"All five seeds meet the end goal at box sides {meeting} m; none does at {none} m. In those larger boxes the flock still passes through the goal circle, "
      f"but no pass lasts the required 30 s. Observed in this setting; the cause has not been established.")

,goal circle's share of the box,seeds meeting the end goal,first met (s),passes per run,longest pass (s),visits of 30 s or more,occupancy after 100 s,cohesion (m),centre to goal (m)
16 m,44%,5 of 5,15 to 103,50.6,77,4 to 7,0.84,2.4,3.1
20 m,28%,5 of 5,56 to 174,36.8,62,3 to 7,0.77,2.7,3.4
24 m,20%,5 of 5,23 to 620,38.2,58,2 to 7,0.66,2.8,4.4
26 m,17%,1 of 5,340 to 340,47.4,34,0 to 2,0.59,3.1,4.9
28 m,14%,0 of 5,never,30.6,20,0 to 0,0.42,3.2,7.1
30 m,13%,0 of 5,never,22.0,21,0 to 0,0.33,3.1,8.5
36 m,9%,0 of 5,never,16.0,11,0 to 0,0.25,3.1,10.7
48 m,5%,0 of 5,never,10.8,11,0 to 0,0.18,3.3,14.2
60 m,3%,0 of 5,never,8.0,10,0 to 0,0.15,4.0,16.4
80 m,2%,0 of 5,never,6.0,11,0 to 0,0.1,7.8,20.7


All five seeds meet the end goal at box sides [16, 20, 24] m; none does at [28, 30, 36, 48, 60, 80] m. In those larger boxes the flock still passes through the goal circle, but no pass lasts the required 30 s. Observed in this setting; the cause has not been established.


## 3. Hybrid A: designed herders, always-walking Jadhav-style sheep

`hybrid_jadhav_sheep_dilorenzo_dogs`: 60 heading-rule sheep that are always active, with Brownian D = 0.5 m²/s so the design has noise to confine; 16 Di Lorenzo herders designed for a weak push 0.8 e^(−r/12 m) m/s that these sheep do not have, K_H = 1, capped at 1.5 m/s, one flow sub-step per second (the preset's setting, not their dt = 0.01); 80 m periodic box; goal circle r* = 10 m; 600 s.

**End goal:** at least 80% of the sheep inside the goal circle for 30 s. No seed reaches it.

**Watch for:** the flock splits into groups that do not come back together. Grey is the target density. Play gives the first 90 s, when the groups form, 7 of its 15 seconds.

In [11]:
cfg_a = B.preset("hybrid_jadhav_sheep_dilorenzo_dogs")
a_runs = {s: run(cfg_a, seed=s) for s in range(4)}
r = a_runs[SEED]
D = r.domain
per_frame = np.array([groups(x, D) for x in r.pos_s])
peak = int(r.metrics["occupancy"].argmax())
pa = bridge_panel(r, "60 sheep, 16 herders, 80 m periodic box", scale=20, scale_label="20 m", sheep_px=3.2, dog_px=7)
strips = [occupancy_strip([("occupancy", SHEEP, r)]), strip("Groups of 3 or more sheep (linked within 5 m)", [("groups", SHEEP, per_frame[:, 0])], 0, per_frame[:, 0].max() + 1),
          strip("Largest group's share of the flock", [("largest", SHEEP, per_frame[:, 1])], 0, 1)]
player("Hybrid A", r.t, [pa], strips, [(r.t[peak], "Highest occupancy")], note=f"Seed {SEED}. The end goal is not reached in this run.", play_seconds=PLAY_SECONDS, pace=[(90, 7)])

In [12]:
rows_a = []
for s, r in a_runs.items():
    sc = score(r)
    rows_a.append(dict(seed=s, reached_goal_at=sc["time_to_success"], occupancy_last_instant=r.final()["occupancy"], occupancy_last_150_s=sc["final_occupancy"],
                       groups_mean=float(np.mean([groups(x, r.domain)[0] for x in r.pos_s[::10]])), largest_group_final=sc["largest_group_final"]))
display(pd.DataFrame(rows_a).set_index("seed").round(3))
saved = stored("bridge_diagnostics")["hybrid_a"]["seeds"]
for mine, ref in zip(rows_a, saved):
    check(f"Hybrid A, seed {mine['seed']}", "occupancy at the last instant", mine["occupancy_last_instant"], ref["final_occupancy"], 0.005, "0 to 0.17")
    check(f"Hybrid A, seed {mine['seed']}", "mean number of groups", mine["groups_mean"], ref["groups_mean"], 0.05, "3 to 4")

,reached_goal_at,occupancy_last_instant,occupancy_last_150_s,groups_mean,largest_group_final
seed,,,,,
0,None,0.000,0.003,3.098,0.433
1,None,0.017,0.003,3.787,0.415
2,None,0.167,0.322,4.066,0.364
3,None,0.000,0.001,4.033,0.266


## 4. Hybrid B: Jadhav's dog, density-model sheep

`hybrid_first_order_sheep_strombom_dogs`: 50 first-order sheep in metres (dog push 2.3 e^(−r/5 m) m/s felt at every distance, Morse cohesion, D = 0.12 m²/s, dt = 0.1 s), one Strömbom dog at 1.5 m/s, open plane, Gaussian goal σ = 3 m (goal circle 7.3 m), starts scattered around the goal (sd 8 m), 900 s. The slides use the first 600 s of these same runs; the run continues to 900 s because the seed 0 flock is only pushed back in at t = 674 s.

**End goal:** at least 80% of the sheep inside the goal circle for 30 s.

**Watch for:** the dog gathers the flock and pushes it out again, because its drive point never stops at the goal. Much later it pushes the flock back into the circle, and out again. The slider shows seed 0, which the docs walk through; the table below has all four seeds. Play slows down for the first overshoot and for the return. Grey is the target density; this dog only steers toward its centre.

In [13]:
HYBRID_B_SECONDS = 900  # default 900: the slides use the first 600 s; the seed 0 flock is only pushed back in at t = 674 s
cfg_b = B.preset("hybrid_first_order_sheep_strombom_dogs")
b_runs = {s: run(cfg_b, seed=s, store_every=10, n_steps=int(HYBRID_B_SECONDS / cfg_b[0].dt)) for s in range(4)}  # one frame per second
r = b_runs[SEED]
occ, back = r.metrics["occupancy"], score(r)["time_to_success"]
peak = int(occ[:60].argmax())  # the first pass
empty = peak + int(np.argmax(occ[peak:] == 0)) if (occ[peak:] == 0).any() else None
limit = float(np.ceil(max(np.abs(r.pos_s).max(), np.abs(r.pos_d).max()))) + 3
pb = bridge_panel(r, "50 first-order sheep, 1 Strömbom dog, open plane", half=limit, scale=10, scale_label="10 m", sheep_px=3.2, dog_px=8,
                  trails=[(r.pos_s.mean(axis=1), SHEEP, 1)], dog_ring=None)
strips = [occupancy_strip([("occupancy", SHEEP, r)]), strip("Flock centre to goal (m)", [("distance", SHEEP, r.metrics["centroid_error"])], 0, 1.05 * r.metrics["centroid_error"].max()),
          strip("Cohesion (m)", [("cohesion", SHEEP, r.metrics["cohesion"])], 0, 1.05 * r.metrics["cohesion"].max())]
events = [(r.t[peak], f"First peak ({100 * occ[peak]:.0f}% inside)")] + ([] if empty is None else [(r.t[empty], "Occupancy falls to 0")])
pace = [(40, 7)]
if back is not None and back > 60:
    out = back + int(np.argmax(occ[int(back):] < 0.5))  # frames are 1 s apart
    events += [(back, "Pushed back in: 80% inside for 30 s"), (out, "Out again: below 50% inside")]
    pace = [(40, 4), (back - 14, 7), (back + 66, 11)]
player("Hybrid B", r.t, [pb], strips, events, note=f"Seed {SEED}. All four seeds are in the table below.", play_seconds=PLAY_SECONDS, pace=pace)

In [14]:
rows_b = []
for s, r in b_runs.items():
    w = (r.t >= 100) & (r.t <= 600)  # the slide's window
    occ = r.metrics["occupancy"]
    rows_b.append(dict(seed=s, reached_goal_at=score(r)["time_to_success"], visits_of_30_s_or_more=len(_episodes(occ >= 0.8, r.t, 30.0)),
                       occupancy_100_600_s=occ[w].mean(), occupancy_100_900_s=occ[r.t >= 100].mean(),
                       centre_to_goal_m_100_600_s=r.metrics["centroid_error"][w].mean(), cohesion_m_100_600_s=r.metrics["cohesion"][w].mean()))
frame_b = pd.DataFrame(rows_b).set_index("seed")
display(frame_b.round(3))
saved = stored("hybrid_b_herders")["rows"]["strombom, plane"]
check("Hybrid B", "occupancy over 100-600 s, mean of 4 seeds", frame_b["occupancy_100_600_s"].mean(), saved["occupancy"]["mean"], 0.005, "0.20")
check("Hybrid B", "centre to goal over 100-600 s (m)", frame_b["centre_to_goal_m_100_600_s"].mean(), saved["centroid_dist_mean"]["mean"], 0.1, "not on slide")

,reached_goal_at,visits_of_30_s_or_more,occupancy_100_600_s,occupancy_100_900_s,centre_to_goal_m_100_600_s,cohesion_m_100_600_s
seed,,,,,,
0,674.0,2,0.001,0.165,18.570,2.389
1,7.0,5,0.281,0.342,11.816,1.682
2,13.0,5,0.257,0.342,9.176,2.433
3,342.0,3,0.250,0.284,12.073,2.324


## 5. This notebook against the slides

`notebook` is what the cells above just computed, `stored` is the value in `results/`, and `slide` is what the deck says. Each row belongs to its own setting; the rows are not a comparison between hybrids.

In [15]:
checks = pd.DataFrame(CHECKS)
print(f"{(checks.match == 'yes').sum()} of {len(checks)} numbers match the stored results.")
display(checks)

19 of 19 numbers match the stored results.


,simulation,quantity,notebook,stored,slide,match
0,Jadhav,mean distance from target after reaching (m),19.641,19.641,about 20 m,yes
1,"Follower swap, Native followers",occupancy,0.859,0.859,86%,yes
2,"Follower swap, Native followers",density error,0.240,0.240,0.24,yes
3,"Follower swap, Native followers",centroid error (m),0.677,0.677,0.7 m,yes
4,"Follower swap, Jadhav sheep",occupancy,1.000,1.000,100%,yes
5,"Follower swap, Jadhav sheep",density error,1.288,1.288,1.29,yes
6,"Follower swap, Jadhav sheep",centroid error (m),0.440,0.440,0.4 m,yes
7,"Follower swap, flat design",occupancy,0.173,0.173,17%,yes
8,"Follower swap, flat design",centroid error (m),10.031,10.031,10 m,yes
9,"Hybrid A, seed 0",occupancy at the last instant,0.000,0.000,0 to 0.17,yes
